# Predicting Income with Folktables + SHAP

This notebook walks through a full ML interpretability workflow on US Census data:

1. Load the ACS Income classification task via `folktables`
2. Train a **Random Forest** classifier
3. Explain predictions using **SHAP** (SHapley Additive exPlanations)
4. Swap in a **LightGBM** model and compare explanations

The prediction target is whether an individual's annual income exceeds $50,000, using 10 demographic and labor-market features from the 2018 American Community Survey (Arizona sample).


In [1]:

try: import folktables
except ImportError:
    !pip install folktables -q
    import folktables

try: import shap
except ImportError:
    !pip install shap -q
    import shap

try: import lightgbm
except ImportError:     
    !pip install lightgbm -q
    import lightgbm

## Imports

We use:
- `folktables` — a benchmark library that wraps ACS microdata into ML-ready classification tasks
- `sklearn` — for the Random Forest model and train/test split
- `shap` — for model explanations
- `lightgbm` — gradient boosted trees (used later as a comparison model)


In [ ]:
from folktables import ACSDataSource, ACSIncome
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import shap as shap
from sklearn.metrics import classification_report
import pandas as pd
import matplotlib.pyplot as plt

## Load Data

`folktables` provides a clean interface to the ACS PUMS (Public Use Microdata Sample). We pull the **ACSIncome** task, which is pre-defined to predict whether income exceeds $50k.

`ACSIncome.df_to_numpy()` applies the task's built-in filtering and feature selection, returning a NumPy feature matrix and a binary label array.


In [ ]:
# Get ACS data for Arizona, 2018
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
acs_data = data_source.get_data(states=["AZ"], download=True)

# Extract features and labels using folktables schema
features, labels, _ = ACSIncome.df_to_numpy(acs_data)

# Turn into DataFrame for readability
X = pd.DataFrame(features, columns=ACSIncome.features)
y = pd.Series(labels)

## Exploratory Data Analysis

Let's get a quick sense of the feature distributions before modeling.


In [ ]:
# EDA for X
print(X.describe())


In [ ]:
# Feature names
feature_names = X.columns.tolist()
print("Feature names:", feature_names)

### Feature Dictionary

The 10 ACS features used in the `ACSIncome` task:


| Column Name | Description |
|-------------|-------------|
| `AGEP`      | Age of the person |
| `COW`       | Class of worker (e.g., private, government, self-employed) |
| `SCHL`      | Highest education level completed |
| `MAR`       | Marital status |
| `OCCP`      | Occupation code |
| `POBP`      | Place of birth (e.g., US state or foreign country) |
| `RELP`      | Relationship to the head of household |
| `WKHP`      | Usual hours worked per week |
| `SEX`       | Sex (0 = male, 1 = female in Folktables) |
| `RAC1P`     | Race (coded — requires mapping for labels) |

We rename the columns from raw ACS codes to human-readable labels for cleaner plots.


In [ ]:
X.columns = [
    "Age", "Class of Work", "Education Level", "Marital Status",
    "Occupation", "Birthplace", "Household Role", "Hours/Week",
    "Sex", "Race"
]

### Label Distribution

The target `y` is binary: `True` if income > $50k, `False` otherwise. It's worth checking class balance — imbalanced classes can bias both the model and the SHAP values.


In [ ]:
print(y.value_counts())

## Model Training: Random Forest

We split 70/30 into train and test sets, then fit a `RandomForestClassifier`. The hyperparameters here are intentionally modest (`n_estimators=20`, `max_depth=8`) to keep training fast for a demo — in practice you'd tune these.

Random Forests are a natural starting point for SHAP analysis because SHAP's `TreeExplainer` handles them exactly (no approximations needed).


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = RandomForestClassifier(n_estimators=20, max_depth=8, random_state=0)
model.fit(X_train, y_train)

## SHAP Explanations

### What is SHAP?

SHAP (SHapley Additive exPlanations) assigns each feature a contribution value for each individual prediction. The value answers: *how much did this feature push the prediction higher or lower compared to the average prediction?*

SHAP values are grounded in cooperative game theory (Shapley values), which guarantees they satisfy desirable properties like consistency and local accuracy.

**Key idea:** for a given sample, the SHAP values for all features sum to `model_output - E[model_output]`. So they're not just ranked importances — they're signed, additive contributions to each prediction.

### Computing SHAP Values

`shap.Explainer` auto-detects the model type and dispatches to `TreeExplainer` for tree models, which computes exact Shapley values efficiently.

For a binary `RandomForestClassifier`, the explainer returns a **3D** Explanation object shaped `(n_samples, n_features, n_classes)` — one set of SHAP values per class. We slice `[:, :, 1]` to work with the positive class (income > $50k).


In [ ]:
# this takes a long time to run
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

# For binary RandomForestClassifier, shap.Explainer returns a 3D Explanation
# object with shape (n_samples, n_features, n_classes).
# Slice [:, :, 1] to get SHAP values for the positive class (income > $50k).
shap_values_pos = shap_values[:, :, 1]


### Summary Plot (Beeswarm)

The beeswarm plot is the most information-dense SHAP visualization. Each dot is one test sample. For each feature (y-axis):

- **x position** = SHAP value (positive = pushed toward high income, negative = pushed toward low income)
- **color** = actual feature value (red = high, blue = low)

Reading patterns:
- A feature with dots spread far left and right has high variance in its impact.
- If red dots cluster on the right, high values of that feature correlate with higher predicted income.


In [ ]:
# Summary plot (beeswarm) — shows feature impact on positive class
shap.summary_plot(shap_values_pos, X_test)


### Summary Plot (Bar)

The bar plot shows **mean absolute SHAP value** per feature — a single-number summary of how much each feature contributes to predictions on average across the test set. This is a global feature importance measure that's more trustworthy than the built-in `feature_importances_` from sklearn, which can be biased toward high-cardinality features.


In [ ]:
# Bar plot — mean absolute SHAP values (overall feature importance)
shap.summary_plot(shap_values_pos, X_test, plot_type="bar")


### Dependence Plots

A dependence plot shows the relationship between a single feature's value (x-axis) and its SHAP value (y-axis) across all test samples. The color encodes an automatically selected interaction feature.

This lets you see *how* a feature affects the model — is the relationship linear? Is there a threshold effect? Does the impact vary by another feature (the colored interaction)?


In [ ]:
# dependence_plot expects a 2D numpy array, so pass shap_values_pos.values
shap.dependence_plot("Education Level", shap_values_pos.values, X_test)


In [ ]:
shap.dependence_plot("Age", shap_values_pos.values, X_test)


## LightGBM Comparison

LightGBM is a gradient boosted tree framework from Microsoft. Key differences from Random Forest:

- **Sequential vs. parallel:** LightGBM builds trees one at a time, each correcting the previous one's residuals. Random Forest builds all trees independently and averages.
- **Leaf-wise growth:** LightGBM splits the highest-loss leaf at each step rather than growing level-by-level, often giving better accuracy per leaf.
- **Speed:** LightGBM's native tree structure is directly readable by SHAP's `TreeExplainer`, making SHAP computation fast.

For binary LightGBM, `shap.Explainer` returns a **2D** Explanation object directly (no slicing needed), so we can pass `shap_values` straight to `summary_plot`.


In [ ]:

model = lgb.LGBMClassifier(n_estimators=50)
model.fit(X_train, y_train)

explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)
shap.summary_plot(shap_values, X_test)